In [1]:
# =============================================================================
# CELL 1 — Imports
# =============================================================================

import cv2
import numpy as np
import re
import logging
from pathlib import Path
from typing import List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision.models as tv_models
import torchvision.transforms as T
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

# OPT-1: Import numba for JIT-compiled RLE (eliminates Python row loop)
try:
    from numba import njit, prange
    HAS_NUMBA = True
except ImportError:
    HAS_NUMBA = False
    logging.warning("numba not found — pip install numba. Falling back to numpy RLE.")

try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False
    logging.warning("imbalanced-learn not found — pip install imbalanced-learn. "
                    "Falling back to repeat oversampling.")

import time
from tqdm import tqdm
from functools import partial

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger(__name__)


# =============================================================================
# CELL 2 — Constants
# =============================================================================

BENIGN_DIR    = r"D:\cancer-ultra sound\breast3\Dataset_BUSI_with_GT\benign"
MALIGNANT_DIR = r"D:\cancer-ultra sound\breast3\Dataset_BUSI_with_GT\malignant"
NORMAL_DIR    = r"D:\cancer-ultra sound\breast3\Dataset_BUSI_with_GT\normal"

CLASS_NAMES   = ["cancer(0)", "normal(1)"]

IMAGE_SIZE     = 256
MAX_RUNS       = 18
NEARFIELD_SKIP = 0.08
FARFIELD_SKIP  = 0.25

Y_START   = int(IMAGE_SIZE * NEARFIELD_SKIP)
Y_END     = int(IMAGE_SIZE * (1.0 - FARFIELD_SKIP))
CROP_ROWS = Y_END - Y_START

ROW_FEAT    = CROP_ROWS   * MAX_RUNS * 2
COL_FEAT    = IMAGE_SIZE  * MAX_RUNS * 2
FEATURE_DIM = ROW_FEAT + COL_FEAT

SEED        = 42
BATCH_SIZE  = 64
EPOCHS      = 200
LR          = 7e-4
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMBALANCE_STRATEGY = "smote"
LOSS_FN            = "focal"
FOCAL_GAMMA        = 2.0
THRESH_K           = 0.6

AUG_PARAMS = dict(
    flip_prob    = 0.00,
    rotate_prob  = 0.00,
    rotate_limit = 4,
)

SE_REDUCTION = 8

MAIN_ATTN_CHANNELS = 128
PAR_ATTN_CHANNELS  = 32
PAR_FUSED_CHANNELS = PAR_ATTN_CHANNELS * 4

# OPT-2: Number of DataLoader workers — use physical cores, cap at 4
# Set 0 on Windows if you hit issues with spawn context
NUM_DATALOADER_WORKERS = 0

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
# =============================================================================
# CELL 3 — Helpers: file listing and mask detection
# =============================================================================

def _is_mask(path: Path) -> bool:
    stem = path.stem
    return stem.endswith("_mask") or bool(re.search(r"_mask_\d+$", stem))


def _get_index(stem: str) -> int:
    m = re.search(r'\((\d+)\)', stem)
    return int(m.group(1)) if m else -1


# =============================================================================
# CELL 4 — Core: RLE feature extraction
# OPT-3: numba @njit replaces Python for-loop over rows.
#         prange enables automatic parallel execution across rows.
#         Falls back to pure-numpy if numba is unavailable.
# =============================================================================

if HAS_NUMBA:
    @njit(parallel=True, cache=True)
    def _rle_numba(binary: np.ndarray, max_runs: int) -> np.ndarray:
        """
        JIT-compiled RLE: processes all rows in parallel via prange.
        ~10-20x faster than the Python for-loop version.
        """
        H, W = binary.shape
        out  = np.zeros((H, max_runs * 2), dtype=np.float32)

        for i in prange(H):                    # OPT: parallel across rows
            col    = 0
            in_run = False
            start  = 0
            run_n  = 0

            for j in range(W):
                val = binary[i, j]
                if val == 1 and not in_run:
                    start  = j
                    in_run = True
                elif val == 0 and in_run:
                    if run_n < max_runs:
                        out[i, run_n * 2]     = start / W
                        out[i, run_n * 2 + 1] = (j - start) / W
                        run_n += 1
                    in_run = False

            # handle run reaching end of row
            if in_run and run_n < max_runs:
                out[i, run_n * 2]     = start / W
                out[i, run_n * 2 + 1] = (W - start) / W

        return out

    def vectorized_rle(binary: np.ndarray,
                        max_runs: int = MAX_RUNS) -> np.ndarray:
        # Ensure C-contiguous for numba
        return _rle_numba(np.ascontiguousarray(binary), max_runs)

else:
    # Pure-numpy fallback (same as original but kept tidy)
    def vectorized_rle(binary: np.ndarray,
                        max_runs: int = MAX_RUNS) -> np.ndarray:
        H, W = binary.shape
        out  = np.zeros((H, max_runs * 2), dtype=np.float32)

        padded = np.concatenate([
            np.zeros((H, 1), dtype=np.int8),
            binary.astype(np.int8),
            np.zeros((H, 1), dtype=np.int8),
        ], axis=1)

        diff = np.diff(padded, axis=1)

        for i in range(H):
            row    = diff[i]
            starts = np.where(row ==  1)[0]
            ends   = np.where(row == -1)[0]
            n = min(len(starts), max_runs)
            if n == 0:
                continue
            lengths = (ends[:n] - starts[:n]).astype(np.float32)
            out[i, 0:n*2:2] = starts[:n].astype(np.float32) / W
            out[i, 1:n*2:2] = lengths / W

        return out


def image_to_features(path: Path,
                       k: float = THRESH_K) -> Optional[np.ndarray]:
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        log.warning("Cannot load %s", path.name)
        return None

    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE),
                     interpolation=cv2.INTER_AREA)
    img = img[Y_START:Y_END, :]

    img_f     = img.astype(np.float32)
    mean      = img_f.mean()
    std       = img_f.std()
    threshold = mean - k * std

    binary = (img_f < threshold).astype(np.uint8)

    rle_row    = vectorized_rle(binary, MAX_RUNS)
    binary_col = binary.T
    rle_col    = vectorized_rle(binary_col, MAX_RUNS)

    return np.concatenate([
        rle_row.flatten(),
        rle_col.flatten()
    ])


# =============================================================================
# CELL 4b — Augmentation helpers
# =============================================================================

def _augment_image(img: np.ndarray,
                   p: dict = AUG_PARAMS) -> list:
    results = []

    if np.random.random() < p['flip_prob']:
        results.append(np.fliplr(img))

    if np.random.random() < p['rotate_prob']:
        h, w    = img.shape
        angle   = np.random.uniform(-p['rotate_limit'],
                                     p['rotate_limit'])
        M       = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        rotated = cv2.warpAffine(img, M, (w, h),
                                 borderMode=cv2.BORDER_REFLECT)
        results.append(rotated)

    return results


def _augment_image_fast(img: np.ndarray,
                         p: dict = AUG_PARAMS) -> list:
    """Faster augmentation with early exit when all probs are zero."""
    results = []

    if p['flip_prob'] == 0.0 and p['rotate_prob'] == 0.0:
        return results                                 # OPT: early exit

    if p['rotate_prob'] > 0 and np.random.random() < p['rotate_prob']:
        h, w  = img.shape
        angle = np.random.uniform(-p['rotate_limit'], p['rotate_limit'])
        M     = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        rotated = cv2.warpAffine(img, M, (w, h),
                                 flags=cv2.INTER_LINEAR,
                                 borderMode=cv2.BORDER_CONSTANT,
                                 borderValue=0)
        results.append(rotated)

    if p['flip_prob'] > 0 and np.random.random() < p['flip_prob']:
        results.append(np.fliplr(img))

    return results


# =============================================================================
# CELL 6 — Build train / test split
# =============================================================================

def load_class_paths(class_dir: Path,
                     pct: float = 0.80
                     ) -> Tuple[List[Path], List[Path]]:

    all_paths = []
    for p in sorted(class_dir.iterdir()):
        if (not p.is_file() or _is_mask(p)
                or p.suffix.lower() not in {'.png', '.jpg', '.jpeg', '.bmp'}):
            continue
        all_paths.append(p)

    if len(all_paths) == 0:
        log.warning("No images found in %s", class_dir.name)
        return [], []

    train_paths, test_paths = train_test_split(
        all_paths,
        test_size    = 1 - pct,
        random_state = SEED,
        shuffle      = True
    )

    log.info("%s | train=%d  test=%d",
             class_dir.name, len(train_paths), len(test_paths))
    return train_paths, test_paths


# =============================================================================
# CELL 7 — Pre-extract all features
# OPT-4: _rle_from_array_optimized uses the fast vectorized_rle (numba if available)
# OPT-5: extract_single_image kept as top-level for pickling compatibility
# =============================================================================

CACHE_DIR = Path("./cache")
CACHE_DIR.mkdir(exist_ok=True)

# OPT-6: Single shared pool size — use physical cores, avoid over-subscription
NUM_WORKERS = min(mp.cpu_count(), 8)


def _rle_from_array_optimized(img_arr: np.ndarray) -> np.ndarray:
    """Convert cropped grayscale array → flat RLE feature vector."""
    img_f     = img_arr.astype(np.float32)
    mean      = img_f.mean()
    std       = img_f.std()
    threshold = mean - THRESH_K * std
    binary    = (img_f < threshold).astype(np.uint8)

    rle_row    = vectorized_rle(binary,     MAX_RUNS)
    binary_col = np.ascontiguousarray(binary.T)   # OPT: ensure C-contiguous before numba
    rle_col    = vectorized_rle(binary_col, MAX_RUNS)

    return np.concatenate([rle_row.flatten(), rle_col.flatten()])


# Legacy alias
def _rle_from_array(img_arr: np.ndarray) -> np.ndarray:
    return _rle_from_array_optimized(img_arr)


def extract_single_image(args):
    """
    Top-level function — required for ProcessPoolExecutor pickling.
    Returns list of (feature_float32, label) tuples.

    OPT-7: Returns float32 directly (avoids float16→float32 cast in Dataset).
    OPT-8: Early augmentation exit when all probs are zero.
    """
    path, label, is_train = args
    results = []

    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return results

    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
    img = img[Y_START:Y_END, :]

    f = _rle_from_array_optimized(img)
    results.append((f.astype(np.float32), label))    # OPT-7: float32

    if is_train and (AUG_PARAMS['flip_prob'] > 0 or AUG_PARAMS['rotate_prob'] > 0):
        for aug_img in _augment_image_fast(img):
            fa = _rle_from_array_optimized(aug_img)
            results.append((fa.astype(np.float32), label))

    return results


# =============================================================================
# OPT-9: Parallel extraction creates pool ONCE (not once per batch).
#         This eliminates repeated process-spawn overhead.
# =============================================================================

def _extract_parallel(paths: list, label: int, is_train: bool):
    """
    Build one process pool for all paths — pool is NOT recreated per batch.
    """
    if not paths:
        return [], []

    args = [(p, label, is_train) for p in paths]

    feats, labels_list = [], []

    # OPT-9: Single pool for all images — no per-batch spawn overhead
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        for result in tqdm(
            executor.map(extract_single_image, args, chunksize=16),  # OPT-10: chunksize reduces IPC
            total=len(args),
            desc=f"  label={label} {'train' if is_train else 'test '}",
        ):
            for f, lbl in result:
                feats.append(f)
                labels_list.append(lbl)

    return feats, labels_list


def extract_train_parallel(paths: list, label: int):
    return _extract_parallel(paths, label, is_train=True)


def extract_test_parallel(paths: list, label: int):
    return _extract_parallel(paths, label, is_train=False)


# =============================================================================
# Sequential fallback functions
# =============================================================================

def extract_train_sequential(paths, label):
    feats, labels = [], []
    for p in tqdm(paths, desc=f"  label={label} train"):
        f = image_to_features(p)
        if f is not None:
            feats.append(f.astype(np.float32))
            labels.append(label)

        if AUG_PARAMS['flip_prob'] > 0 or AUG_PARAMS['rotate_prob'] > 0:
            img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE),
                             interpolation=cv2.INTER_AREA)
            img = img[Y_START:Y_END, :]
            for aug_img in _augment_image(img):
                fa = _rle_from_array_optimized(aug_img)
                feats.append(fa.astype(np.float32))
                labels.append(label)
    return feats, labels


def extract_test_sequential(paths, label):
    feats, labels = [], []
    for p in tqdm(paths, desc=f"  label={label} test"):
        f = image_to_features(p)
        if f is not None:
            feats.append(f.astype(np.float32))
            labels.append(label)
    return feats, labels


# =============================================================================
# CELL 7b — build_dataset with caching
# OPT-11: Save/load cache as float32 (no more float16 downcasting)
# =============================================================================

def build_dataset(force_recompute=False, use_parallel=True):
    cache_files = {
        'X_train': CACHE_DIR / "X_train.npy",
        'X_test' : CACHE_DIR / "X_test.npy",
        'y_train': CACHE_DIR / "y_train.npy",
        'y_test' : CACHE_DIR / "y_test.npy",
    }

    if not force_recompute and all(f.exists() for f in cache_files.values()):
        print("\n" + "="*60)
        print("LOADING FROM CACHE")
        print("="*60)

        X_train = np.load(cache_files['X_train'])
        X_test  = np.load(cache_files['X_test'])
        y_train = np.load(cache_files['y_train'])
        y_test  = np.load(cache_files['y_test'])

        print(f"  Train: {X_train.shape}  cancer={int((y_train==0).sum())}  normal={int((y_train==1).sum())}")
        print(f"  Test:  {X_test.shape}   cancer={int((y_test==0).sum())}  normal={int((y_test==1).sum())}")
        print(f"  dtype={X_train.dtype}  {X_train.nbytes/1024**2:.1f} MB")
        return X_train, X_test, y_train, y_test

    print("\n" + "="*60)
    print("EXTRACTING FEATURES FROM SCRATCH")
    print("="*60)
    print(f"Augmentation: flip={AUG_PARAMS['flip_prob']}  rotate={AUG_PARAMS['rotate_prob']}")

    b_train, b_test = load_class_paths(Path(BENIGN_DIR))
    m_train, m_test = load_class_paths(Path(MALIGNANT_DIR))
    n_train, n_test = load_class_paths(Path(NORMAL_DIR))

    original_train_count = len(b_train) + len(m_train) + len(n_train)
    print(f"\n[1/4] Extracting from {original_train_count} training images...")

    extract_fn_tr = extract_train_parallel  if use_parallel else extract_train_sequential
    extract_fn_te = extract_test_parallel   if use_parallel else extract_test_sequential

    print("\n  → Benign (cancer) train")
    bf,   bl   = extract_fn_tr(b_train, 0)
    print("  → Malignant (cancer) train")
    mf,   ml   = extract_fn_tr(m_train, 0)
    print("  → Normal train")
    nf,   nl   = extract_fn_tr(n_train, 1)
    print("  → Benign (cancer) test")
    bf_t, bl_t = extract_fn_te(b_test,  0)
    print("  → Malignant (cancer) test")
    mf_t, ml_t = extract_fn_te(m_test,  0)
    print("  → Normal test")
    nf_t, nl_t = extract_fn_te(n_test,  1)

    print("\n[2/4] Combining...")
    X_train = np.array(bf + mf + nf,          dtype=np.float32)   # OPT-11: float32
    y_train = np.array(bl + ml + nl,          dtype=np.int64)
    X_test  = np.array(bf_t + mf_t + nf_t,   dtype=np.float32)
    y_test  = np.array(bl_t + ml_t + nl_t,   dtype=np.int64)

    print("\n[3/4] Saving cache...")
    np.save(cache_files['X_train'], X_train)
    np.save(cache_files['X_test'],  X_test)
    np.save(cache_files['y_train'], y_train)
    np.save(cache_files['y_test'],  y_test)

    actual = len(X_train)
    print(f"\n[4/4] Stats:")
    print(f"  Original train images : {original_train_count}")
    print(f"  Actual train samples  : {actual}  ({actual/original_train_count:.2f}x)")
    print(f"  Train shape: {X_train.shape}  cancer={int((y_train==0).sum())}  normal={int((y_train==1).sum())}")
    print(f"  Test  shape: {X_test.shape}")
    print(f"  Memory: {X_train.nbytes/1024**2:.1f} MB (train)  {X_test.nbytes/1024**2:.1f} MB (test)")

    return X_train, X_test, y_train, y_test


# =============================================================================
# CELL 8 — Imbalance handling utilities
# =============================================================================

def apply_oversampling(X: np.ndarray,
                       y: np.ndarray,
                       strategy: str = IMBALANCE_STRATEGY
                       ) -> Tuple[np.ndarray, np.ndarray]:
    n0 = int((y == 0).sum())
    n1 = int((y == 1).sum())
    minority = 0 if n0 < n1 else 1
    majority = 1 - minority

    print(f"\n  Class distribution before resampling: "
          f"cancer(0)={n0}  normal(1)={n1}  "
          f"(minority=class {minority})")

    if strategy in ("smote", "oversample"):
        if strategy == "smote":
            print("  [SMOTE disabled for RLE] "
                  "using exact-fill repeat oversampling instead.")
        X_r, y_r = _repeat_oversample(X, y, minority, majority)
    else:
        print("  No resampling (strategy='none').")
        X_r, y_r = X, y

    perm = np.random.default_rng(SEED).permutation(len(X_r))
    return X_r[perm], y_r[perm]


def _repeat_oversample(X: np.ndarray,
                        y: np.ndarray,
                        minority_cls: int,
                        majority_cls: int) -> Tuple[np.ndarray, np.ndarray]:
    min_idx = np.where(y == minority_cls)[0]
    maj_idx = np.where(y == majority_cls)[0]
    deficit = len(maj_idx) - len(min_idx)

    rng      = np.random.default_rng(SEED)
    fill_idx = rng.choice(min_idx, size=deficit, replace=True)

    X_r = np.concatenate([X, X[fill_idx]], axis=0)
    y_r = np.concatenate([y, y[fill_idx]], axis=0)

    print(f"  Exact-fill oversample (+{deficit} samples) → "
          f"cancer(0)={int((y_r == 0).sum())}  "
          f"normal(1)={int((y_r == 1).sum())}")
    return X_r, y_r


def compute_loss_fn(y: np.ndarray) -> nn.Module:
    classes = np.array([0, 1])
    weights = compute_class_weight('balanced', classes=classes, y=y)
    cw      = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    print(f"  Class weights — cancer(0): {weights[0]:.3f}  "
          f"normal(1): {weights[1]:.3f}")
    if LOSS_FN == "focal":
        return FocalLoss(alpha=cw, gamma=FOCAL_GAMMA)
    else:
        return nn.CrossEntropyLoss(weight=cw)


def make_weighted_sampler(y: np.ndarray) -> WeightedRandomSampler:
    class_counts   = np.bincount(y)
    sample_weights = 1.0 / class_counts[y]
    generator      = torch.Generator()
    generator.manual_seed(SEED)
    return WeightedRandomSampler(
        weights     = torch.from_numpy(sample_weights).float(),
        num_samples = len(sample_weights),
        replacement = True,
        generator   = generator,
    )


# =============================================================================
# CELL 8b — Focal Loss
# =============================================================================

class FocalLoss(nn.Module):
    def __init__(self,
                 alpha:     Optional[torch.Tensor] = None,
                 gamma:     float = 2.0,
                 reduction: str   = "mean"):
        super().__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor,
                targets: torch.Tensor) -> torch.Tensor:
        log_prob = F.log_softmax(logits, dim=1)
        prob     = log_prob.exp()
        pt       = prob.gather(1, targets.unsqueeze(1)).squeeze(1)
        ce       = F.nll_loss(log_prob, targets, reduction="none")
        focal_w  = (1.0 - pt) ** self.gamma
        loss     = focal_w * ce

        if self.alpha is not None:
            at   = self.alpha.to(logits.device)[targets]
            loss = at * loss

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


# =============================================================================
# CELL 9 — PyTorch Dataset
# OPT-12: No dtype conversion needed — data is already float32 from extraction.
# =============================================================================

class RLEDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        # OPT-12: from_numpy avoids a copy when dtype already matches
        self.X = torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32))
        self.y = torch.from_numpy(y.astype(np.int64))

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
# =============================================================================
# CELL 9b — SEBlock (Squeeze-and-Excitation)
# =============================================================================

class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (batch, channels)
        w = self.fc(x)
        return x * w

# =============================================================================
# CELL 10 — RLEClassifier with tunable conv parameters
# Edit the CONFIG block below to experiment with different kernels/paddings
# =============================================================================

class RLEClassifier(nn.Module):

    # ┌─────────────────────────────────────────────────────┐
    # │  TUNABLE CONFIG — change these freely               │
    # ├─────────────────────────────────────────────────────┤
    MAIN_K1      = 4     # main branch: first  conv kernel
    MAIN_P1      = 2      # main branch: first  conv padding
    MAIN_K2      = 5      # main branch: second conv kernel
    MAIN_P2      = 2    # main branch: second conv padding

    PAR_KA       = 5     # parallel branch A kernel
    PAR_PA       = 2      # parallel branch A padding
    PAR_KB       = 11     # parallel branch B kernel
    PAR_PB       = 5     # parallel branch B padding
    PAR_KC       = 13     # parallel branch C kernel
    PAR_PC       = 6      # parallel branch C padding
    PAR_KD       = 17     # parallel branch D kernel
    PAR_PD       =8      # parallel branch D padding

    FC_HIDDEN    = 100    # FC hidden dim
    FC_DROP1     = 0.10  # dropout after first FC layer
    # └─────────────────────────────────────────────────────┘

    def __init__(self, num_classes=2):
        super().__init__()

        in_ch = MAX_RUNS * 2

        self.conv_main = nn.Sequential(
    nn.Conv1d(in_ch, 64, kernel_size=self.MAIN_K1, padding='same'),
    nn.BatchNorm1d(64),
    nn.ReLU(inplace=True),
    nn.Dropout(0.05),

    nn.Conv1d(64, MAIN_ATTN_CHANNELS, kernel_size=self.MAIN_K2, padding='same'),
    nn.BatchNorm1d(MAIN_ATTN_CHANNELS),
    nn.ReLU(inplace=True),
    nn.Dropout(0.05),
)

        self.proj = nn.Conv1d(in_ch, MAIN_ATTN_CHANNELS, kernel_size=1, padding='same', bias=False)
        self.gap_main = nn.AdaptiveAvgPool1d(1)
        self.se_a     = SEBlock(channels=MAIN_ATTN_CHANNELS, reduction=SE_REDUCTION)

        self.par_a = nn.Sequential(
            nn.Conv1d(in_ch, PAR_ATTN_CHANNELS, kernel_size=self.PAR_KA, padding=self.PAR_PA),
            nn.BatchNorm1d(PAR_ATTN_CHANNELS), nn.ReLU(inplace=True), nn.Dropout(0.05),
        )
        self.par_b = nn.Sequential(
            nn.Conv1d(in_ch, PAR_ATTN_CHANNELS, kernel_size=self.PAR_KB, padding=self.PAR_PB),
            nn.BatchNorm1d(PAR_ATTN_CHANNELS), nn.ReLU(inplace=True), nn.Dropout(0.05),
        )
        self.par_c = nn.Sequential(
            nn.Conv1d(in_ch, PAR_ATTN_CHANNELS, kernel_size=self.PAR_KC, padding=self.PAR_PC),
            nn.BatchNorm1d(PAR_ATTN_CHANNELS), nn.ReLU(inplace=True), nn.Dropout(0.05),
        )
        self.par_d = nn.Sequential(
            nn.Conv1d(in_ch, PAR_ATTN_CHANNELS, kernel_size=self.PAR_KD, padding=self.PAR_PD),
            nn.BatchNorm1d(PAR_ATTN_CHANNELS), nn.ReLU(inplace=True), nn.Dropout(0.05),
        )

        self.gap_par = nn.AdaptiveAvgPool1d(1)
        self.se_b    = SEBlock(channels=PAR_FUSED_CHANNELS, reduction=SE_REDUCTION)

        self.fc = nn.Sequential(
            nn.Linear(512, self.FC_HIDDEN),
            nn.BatchNorm1d(self.FC_HIDDEN),
            nn.ReLU(inplace=True),
            nn.Dropout(self.FC_DROP1),
            nn.Linear(self.FC_HIDDEN, num_classes),
        )

    def forward(self, x):
        x_row = x[:, :ROW_FEAT].view(x.size(0), CROP_ROWS,  MAX_RUNS * 2)
        x_col = x[:, ROW_FEAT:].view(x.size(0), IMAGE_SIZE, MAX_RUNS * 2)

        seq_row = x_row.permute(0, 2, 1)
        seq_col = x_col.permute(0, 2, 1)

        # NOTE: residual only added when output length matches input length.
        # With AdaptiveAvgPool1d(1) after, length mismatch is harmless for
        # the parallel branches, but for conv_main + proj residual you must
        # ensure MAIN_K1/P1 and MAIN_K2/P2 keep the same length, i.e.:
        #   padding = (kernel_size - 1) // 2   for odd kernels
        #   padding = kernel_size // 2 - 1     for even kernels  (approx)
        # A safe formula for any kernel: padding = kernel_size // 2
        y_row = self.conv_main(seq_row) + self.proj(seq_row)
        y_row = self.gap_main(y_row).squeeze(-1)
        y_row = self.se_a(y_row)

        y_col = self.conv_main(seq_col) + self.proj(seq_col)
        y_col = self.gap_main(y_col).squeeze(-1)
        y_col = self.se_a(y_col)

        a_r = self.gap_par(self.par_a(seq_row)).squeeze(-1)
        b_r = self.gap_par(self.par_b(seq_row)).squeeze(-1)
        c_r = self.gap_par(self.par_c(seq_row)).squeeze(-1)
        d_r = self.gap_par(self.par_d(seq_row)).squeeze(-1)
        p_row = self.se_b(torch.cat([a_r, b_r, c_r, d_r], dim=1))

        a_c = self.gap_par(self.par_a(seq_col)).squeeze(-1)
        b_c = self.gap_par(self.par_b(seq_col)).squeeze(-1)
        c_c = self.gap_par(self.par_c(seq_col)).squeeze(-1)
        d_c = self.gap_par(self.par_d(seq_col)).squeeze(-1)
        p_col = self.se_b(torch.cat([a_c, b_c, c_c, d_c], dim=1))

        fused = torch.cat([y_row, y_col, p_row, p_col], dim=1)
        return self.fc(fused)


# =============================================================================
# CELL 11 — Training loop
# OPT-13: torch.cuda.amp mixed-precision (if CUDA available) — ~2x faster
#          on GPU with no accuracy loss.
# =============================================================================

def train_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device, non_blocking=True)   # OPT-14: non_blocking
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)              # OPT-15: faster zero_grad

        if scaler is not None:                             # OPT-13: AMP forward
            with torch.cuda.amp.autocast():
                logits = model(X_batch)
                loss   = criterion(logits, y_batch)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        total_loss += loss.item() * len(y_batch)
        correct    += (logits.argmax(1) == y_batch).sum().item()
        total      += len(y_batch)

    return total_loss / total, correct / total


# =============================================================================
# CELL 12 — Evaluation loop
# OPT-16: non_blocking transfers, AMP inference
# =============================================================================

def evaluate(model, loader, criterion, device, scaler=None):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device, non_blocking=True)   # OPT-16
            y_batch = y_batch.to(device, non_blocking=True)

            if scaler is not None:
                with torch.cuda.amp.autocast():
                    logits = model(X_batch)
                    loss   = criterion(logits, y_batch)
            else:
                logits = model(X_batch)
                loss   = criterion(logits, y_batch)

            probs = torch.softmax(logits, dim=1)[:, 1]

            total_loss += loss.item() * len(y_batch)
            correct    += (logits.argmax(1) == y_batch).sum().item()
            total      += len(y_batch)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    avg_loss = total_loss / total
    acc      = correct / total
    auc      = roc_auc_score(all_labels, all_probs)
    return avg_loss, acc, auc, np.array(all_labels), np.array(all_probs)


# =============================================================================
# CELL 13 — Full training run
# OPT-17: DataLoader uses NUM_DATALOADER_WORKERS and pin_memory=True
# OPT-18: AMP GradScaler created once and passed to train_epoch / evaluate
# =============================================================================

def train(X_train, X_test, y_train, y_test):

    print(f"\n[Imbalance] strategy='{IMBALANCE_STRATEGY}'  loss='{LOSS_FN}'")

    criterion    = compute_loss_fn(y_train)
    X_bal, y_bal = apply_oversampling(X_train, y_train, IMBALANCE_STRATEGY)

    train_ds = RLEDataset(X_bal,  y_bal)
    test_ds  = RLEDataset(X_test, y_test)

    sampler = make_weighted_sampler(y_bal)

    # OPT-17: persistent_workers avoids re-spawning workers each epoch
    use_workers  = NUM_DATALOADER_WORKERS
    pin          = (DEVICE.type == "cuda")
    persistent_w = (use_workers > 0)

    train_loader = DataLoader(
        train_ds,
        batch_size        = BATCH_SIZE,
        sampler           = sampler,
        num_workers       = use_workers,        # OPT-17
        pin_memory        = pin,                # OPT-17
        persistent_workers= persistent_w,       # OPT-17
        prefetch_factor   = 2 if use_workers > 0 else None,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size        = BATCH_SIZE,
        shuffle           = False,
        num_workers       = use_workers,
        pin_memory        = pin,
        persistent_workers= persistent_w,
        prefetch_factor   = 2 if use_workers > 0 else None,
    )

    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    model     = RLEClassifier(num_classes=2).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

    # OPT-18: AMP scaler — only active on CUDA
    scaler = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None

    best_auc, best_state = 0.0, None
    best_epoch           = 0
    patience             = 60
    epochs_no_improve    = 0

    print(f"\nTraining on {DEVICE}  MAX_RUNS={MAX_RUNS}  SE_reduction={SE_REDUCTION}  "
          f"fused_dim=512  workers={use_workers}  AMP={'on' if scaler else 'off'}")
    print(f"{'Epoch':>5}  {'TrainLoss':>10}  {'TrainAcc':>9}  "
          f"{'ValLoss':>8}  {'ValAcc':>7}  {'AUC':>7}")
    print("-" * 62)

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader,
                                      optimizer, criterion, DEVICE, scaler)
        va_loss, va_acc, va_auc, _, _ = evaluate(model, test_loader,
                                                  criterion, DEVICE, scaler)
        scheduler.step()

        if va_auc > best_auc:
            best_auc          = va_auc
            best_epoch        = epoch
            epochs_no_improve = 0
            best_state        = {k: v.cpu().clone()
                                 for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1

        print(f"{epoch:>5}  {tr_loss:>10.4f}  {tr_acc:>9.4f}  "
              f"{va_loss:>8.4f}  {va_acc:>7.4f}  {va_auc:>7.4f}"
              + (" ✓ best" if epoch == best_epoch else ""))

        if epochs_no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch} "
                  f"(no improvement for {patience} epochs)")
            break

    model.load_state_dict(best_state)
    print(f"\nBest epoch: {best_epoch}  |  Best AUC: {best_auc:.4f}")
    return model, test_loader, criterion


# =============================================================================
# CELL 14 — Shared threshold sweep helper
# =============================================================================

def best_threshold_sweep(y_true: np.ndarray,
                          y_prob: np.ndarray) -> Tuple[float, float]:
    best_thr, best_f1 = 0.5, 0.0
    for thr in np.round(np.arange(0.05, 0.96, 0.01), 2):
        pred = (y_prob >= thr).astype(int)
        f1   = f1_score(y_true, pred)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return best_thr, best_f1


# =============================================================================
# CELL 15 — Final report for RLE model
# =============================================================================

def final_report(model, test_loader, criterion):
    _, _, _, y_true, y_prob = evaluate(model, test_loader, criterion, DEVICE)

    best_thr, best_f1 = best_threshold_sweep(y_true, y_prob)
    y_pred            = (y_prob >= best_thr).astype(int)
    tn, fp, fn, tp    = confusion_matrix(y_true, y_pred).ravel()

    print(f"\n=== Final Report (best threshold = {best_thr:.2f} | "
          f"F1 = {best_f1:.4f} | TP={tp} FP={fp} FN={fn} TN={tn}) ===")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(f"ROC-AUC: {roc_auc_score(y_true, y_prob):.4f}")

    print("\nFull threshold sweep:")
    print(f"{'Thr':>5}  {'F1':>6}  {'TP':>4}  {'FP':>4}  {'FN':>4}  {'TN':>4}  {'Best':>5}")
    print("-" * 48)
    for thr in np.round(np.arange(0.05, 0.96, 0.01), 2):
        pred               = (y_prob >= thr).astype(int)
        f1                 = f1_score(y_true, pred)
        tn_, fp_, fn_, tp_ = confusion_matrix(y_true, pred).ravel()
        marker             = " ←" if thr == best_thr else ""
        print(f"{thr:>5.2f}  {f1:>6.4f}  {tp_:>4}  {fp_:>4}  {fn_:>4}  {tn_:>4}{marker}")


# =============================================================================
# CELL 16 — Traditional ML benchmarks
# =============================================================================

def run_sklearn_benchmarks(X_train, X_test, y_train, y_test):

    X_bal, y_bal = apply_oversampling(X_train, y_train, IMBALANCE_STRATEGY)

    scaler  = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_bal)
    X_te_sc = scaler.transform(X_test)

    pca     = PCA(n_components=150, random_state=SEED)
    X_tr_pc = pca.fit_transform(X_tr_sc)
    X_te_pc = pca.transform(X_te_sc)

    models = {
        "Random Forest"      : (RandomForestClassifier(
                                    n_estimators=300, max_depth=12,
                                    class_weight='balanced',
                                    random_state=SEED, n_jobs=-1),
                                X_bal, X_test),

        "Gradient Boosting"  : (GradientBoostingClassifier(
                                    n_estimators=150, max_depth=4,
                                    learning_rate=0.05,
                                    random_state=SEED),
                                X_bal, X_test),

        "SVM (RBF)"          : (SVC(kernel='rbf', C=2.0, gamma='scale',
                                    class_weight='balanced',
                                    probability=True, random_state=SEED),
                                X_tr_pc, X_te_pc),

        "Logistic Regression": (LogisticRegression(
                                    C=1.0, max_iter=2000,
                                    class_weight='balanced',
                                    random_state=SEED),
                                X_tr_sc, X_te_sc),

        "KNN (k=7)"          : (KNeighborsClassifier(
                                    n_neighbors=7, n_jobs=-1),
                                X_tr_pc, X_te_pc),
    }

    results = {}
    print(f"\n{'Model':<22}  {'Acc':>6}  {'AUC':>6}  {'F1':>6}  "
          f"{'Recall_N':>8}  {'Thr':>5}  {'Time(s)':>8}")
    print("-" * 72)

    for name, (clf, Xtr, Xte) in models.items():
        t0 = time.time()
        clf.fit(Xtr, y_bal)
        elapsed = time.time() - t0

        y_prob             = clf.predict_proba(Xte)[:, 1]
        auc                = roc_auc_score(y_test, y_prob)
        best_thr, best_f1  = best_threshold_sweep(y_test, y_prob)
        y_pred             = (y_prob >= best_thr).astype(int)
        acc                = (y_pred == y_test).mean()
        tn, fp, fn, tp     = confusion_matrix(y_test, y_pred).ravel()
        recall_n           = tp / (tp + fn)

        y_pred_fixed            = (y_prob >= 0.50).astype(int)
        tn_f, fp_f, fn_f, tp_f  = confusion_matrix(y_test, y_pred_fixed).ravel()

        results[name] = dict(
            acc      = acc,
            auc      = auc,
            f1       = best_f1,
            recall_n = recall_n,
            time     = elapsed,
            thr      = best_thr,
            f1_fix   = f1_score(y_test, y_pred_fixed),
            rec_fix  = tp_f / (tp_f + fn_f),
            acc_fix  = (y_pred_fixed == y_test).mean(),
        )

        print(f"{name:<22}  {acc:>6.4f}  {auc:>6.4f}  {best_f1:>6.4f}  "
              f"{recall_n:>8.4f}  {best_thr:>5.2f}  {elapsed:>8.2f}")

    return results


# =============================================================================
# CELL 17 — CNN benchmark
# OPT-19: DataLoader uses workers + pin_memory here too
# =============================================================================

class BenchmarkCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2), nn.Dropout2d(0.3),

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512), nn.BatchNorm1d(512),
            nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(512, 128), nn.BatchNorm1d(128),
            nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class PixelDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths  = paths
        self.labels = labels

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(str(self.paths[idx]), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE),
                         interpolation=cv2.INTER_AREA)
        x   = torch.tensor(img, dtype=torch.float32).unsqueeze(0) / 255.0
        y   = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


def run_cnn_benchmark(benign_dir, malignant_dir, normal_dir, pct=0.80):

    b_train, b_test = load_class_paths(Path(benign_dir),    pct)
    m_train, m_test = load_class_paths(Path(malignant_dir), pct)
    n_train, n_test = load_class_paths(Path(normal_dir),    pct)

    train_paths  = b_train + m_train + n_train
    train_labels = [0]*len(b_train) + [0]*len(m_train) + [1]*len(n_train)
    test_paths   = b_test  + m_test  + n_test
    test_labels  = [0]*len(b_test)  + [0]*len(m_test)  + [1]*len(n_test)

    y_tr_np   = np.array(train_labels)
    sampler   = make_weighted_sampler(y_tr_np)
    criterion = compute_loss_fn(y_tr_np)

    train_ds = PixelDataset(train_paths, train_labels)
    test_ds  = PixelDataset(test_paths,  test_labels)

    pin = (DEVICE.type == "cuda")
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              sampler=sampler,
                              num_workers=NUM_DATALOADER_WORKERS,   # OPT-19
                              pin_memory=pin)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE,
                              shuffle=False,
                              num_workers=NUM_DATALOADER_WORKERS,
                              pin_memory=pin)

    model     = BenchmarkCNN(num_classes=2).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler    = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None

    best_auc, best_state = 0.0, None
    best_epoch           = 0
    patience             = 60
    epochs_no_improve    = 0
    t0 = time.time()

    print(f"\n--- CNN Pixel Baseline (cancer vs normal) ---")
    print(f"{'Epoch':>5}  {'TrainLoss':>10}  {'TrainAcc':>9}  "
          f"{'ValAcc':>7}  {'AUC':>7}")
    print("-" * 50)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        tr_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            if scaler:
                with torch.cuda.amp.autocast():
                    out  = model(xb)
                    loss = criterion(out, yb)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                out  = model(xb)
                loss = criterion(out, yb)
                loss.backward()
                optimizer.step()
            tr_loss += loss.item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            total   += len(yb)
        scheduler.step()

        model.eval()
        all_probs, all_labels_v = [], []
        val_correct, val_total  = 0, 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
                if scaler:
                    with torch.cuda.amp.autocast():
                        out = model(xb)
                else:
                    out = model(xb)
                probs = torch.softmax(out, dim=1)[:, 1]
                val_correct += (out.argmax(1) == yb).sum().item()
                val_total   += len(yb)
                all_probs.extend(probs.cpu().numpy())
                all_labels_v.extend(yb.cpu().numpy())

        va_acc = val_correct / val_total
        va_auc = roc_auc_score(all_labels_v, all_probs)

        if va_auc > best_auc:
            best_auc = va_auc; best_epoch = epoch
            epochs_no_improve = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1

        print(f"{epoch:>5}  {tr_loss/total:>10.4f}  {correct/total:>9.4f}  "
              f"{va_acc:>7.4f}  {va_auc:>7.4f}"
              + (" ✓ best" if epoch == best_epoch else ""))

        if epochs_no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch}")
            break

    elapsed = time.time() - t0
    model.load_state_dict(best_state)

    model.eval()
    all_probs, all_labels_v = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            if scaler:
                with torch.cuda.amp.autocast():
                    probs = torch.softmax(model(xb), dim=1)[:, 1]
            else:
                probs = torch.softmax(model(xb), dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy())
            all_labels_v.extend(yb.numpy())

    y_prob            = np.array(all_probs)
    y_true            = np.array(all_labels_v)
    best_thr, best_f1 = best_threshold_sweep(y_true, y_prob)
    y_pred            = (y_prob >= best_thr).astype(int)
    tn, fp, fn, tp    = confusion_matrix(y_true, y_pred).ravel()
    y_pred_fixed      = (y_prob >= 0.50).astype(int)
    tn_f, fp_f, fn_f, tp_f = confusion_matrix(y_true, y_pred_fixed).ravel()

    print(f"\nCNN AUC: {roc_auc_score(y_true, y_prob):.4f}  best_thr: {best_thr:.2f}")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    return dict(acc=((y_pred==y_true).mean()), auc=roc_auc_score(y_true,y_prob),
                f1=best_f1, recall_n=tp/(tp+fn), time=elapsed, thr=best_thr,
                f1_fix=f1_score(y_true,y_pred_fixed),
                rec_fix=tp_f/(tp_f+fn_f),
                acc_fix=(y_pred_fixed==y_true).mean())


# =============================================================================
# CELL 18 — ResNet-18 benchmark
# OPT-20: Same DataLoader + AMP improvements applied here
# =============================================================================

class ResNetDataset(Dataset):
    def __init__(self, paths, labels, augment=False):
        self.paths  = paths
        self.labels = labels
        mean = [0.485, 0.456, 0.406]
        std  = [0.229, 0.224, 0.225]
        if augment:
            self.tfm = T.Compose([
                T.ToPILImage(), T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                T.RandomRotation(10), T.ToTensor(), T.Normalize(mean, std),
            ])
        else:
            self.tfm = T.Compose([
                T.ToPILImage(), T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                T.ToTensor(), T.Normalize(mean, std),
            ])

    def __len__(self):  return len(self.paths)

    def __getitem__(self, idx):
        img  = cv2.imread(str(self.paths[idx]), cv2.IMREAD_GRAYSCALE)
        img  = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE),
                          interpolation=cv2.INTER_AREA)
        img3 = np.stack([img, img, img], axis=-1)
        x    = self.tfm(img3)
        y    = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


def run_resnet_benchmark(benign_dir, malignant_dir, normal_dir, pct=0.80):

    b_train, b_test = load_class_paths(Path(benign_dir),    pct)
    m_train, m_test = load_class_paths(Path(malignant_dir), pct)
    n_train, n_test = load_class_paths(Path(normal_dir),    pct)

    train_paths  = b_train + m_train + n_train
    train_labels = [0]*len(b_train) + [0]*len(m_train) + [1]*len(n_train)
    test_paths   = b_test  + m_test  + n_test
    test_labels  = [0]*len(b_test)  + [0]*len(m_test)  + [1]*len(n_test)

    y_tr_np   = np.array(train_labels)
    sampler   = make_weighted_sampler(y_tr_np)
    criterion = compute_loss_fn(y_tr_np)

    train_ds = ResNetDataset(train_paths, train_labels, augment=True)
    test_ds  = ResNetDataset(test_paths,  test_labels,  augment=False)

    pin = (DEVICE.type == "cuda")
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              sampler=sampler,
                              num_workers=NUM_DATALOADER_WORKERS,   # OPT-20
                              pin_memory=pin)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE,
                              shuffle=False,
                              num_workers=NUM_DATALOADER_WORKERS,
                              pin_memory=pin)

    model    = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, 2)
    model    = model.to(DEVICE)

    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler    = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None

    best_auc, best_state = 0.0, None
    best_epoch           = 0
    patience             = 60
    epochs_no_improve    = 0
    t0                   = time.time()

    print(f"\n--- ResNet-18 (ImageNet, cancer vs normal) ---")
    print(f"{'Epoch':>5}  {'TrainLoss':>10}  {'TrainAcc':>9}  "
          f"{'ValAcc':>7}  {'AUC':>7}")
    print("-" * 50)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        tr_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            if scaler:
                with torch.cuda.amp.autocast():
                    out  = model(xb)
                    loss = criterion(out, yb)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                out  = model(xb)
                loss = criterion(out, yb)
                loss.backward()
                optimizer.step()
            tr_loss += loss.item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            total   += len(yb)
        scheduler.step()

        model.eval()
        all_probs, all_labels_v = [], []
        val_correct, val_total  = 0, 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
                if scaler:
                    with torch.cuda.amp.autocast():
                        out = model(xb)
                else:
                    out = model(xb)
                probs = torch.softmax(out, dim=1)[:, 1]
                val_correct += (out.argmax(1) == yb).sum().item()
                val_total   += len(yb)
                all_probs.extend(probs.cpu().numpy())
                all_labels_v.extend(yb.cpu().numpy())

        va_acc = val_correct / val_total
        va_auc = roc_auc_score(all_labels_v, all_probs)

        if va_auc > best_auc:
            best_auc = va_auc; best_epoch = epoch
            epochs_no_improve = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1

        print(f"{epoch:>5}  {tr_loss/total:>10.4f}  {correct/total:>9.4f}  "
              f"{va_acc:>7.4f}  {va_auc:>7.4f}"
              + (" ✓ best" if epoch == best_epoch else ""))

        if epochs_no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch}")
            break

    elapsed = time.time() - t0
    model.load_state_dict(best_state)

    model.eval()
    all_probs, all_labels_v = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            if scaler:
                with torch.cuda.amp.autocast():
                    probs = torch.softmax(model(xb), dim=1)[:, 1]
            else:
                probs = torch.softmax(model(xb), dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy())
            all_labels_v.extend(yb.numpy())

    y_prob            = np.array(all_probs)
    y_true            = np.array(all_labels_v)
    best_thr, best_f1 = best_threshold_sweep(y_true, y_prob)
    y_pred            = (y_prob >= best_thr).astype(int)
    tn, fp, fn, tp    = confusion_matrix(y_true, y_pred).ravel()
    y_pred_fixed      = (y_prob >= 0.50).astype(int)
    tn_f, fp_f, fn_f, tp_f = confusion_matrix(y_true, y_pred_fixed).ravel()

    print(f"\nResNet AUC: {roc_auc_score(y_true,y_prob):.4f}  best_thr: {best_thr:.2f}")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    return dict(acc=((y_pred==y_true).mean()), auc=roc_auc_score(y_true,y_prob),
                f1=best_f1, recall_n=tp/(tp+fn), time=elapsed, thr=best_thr,
                f1_fix=f1_score(y_true,y_pred_fixed),
                rec_fix=tp_f/(tp_f+fn_f),
                acc_fix=(y_pred_fixed==y_true).mean())


# =============================================================================
# CELL 19 — Comparison table
# =============================================================================

def print_comparison(sk_results, cnn_result, rle_result, resnet_result):

    # ----------------------------------------------------------------
    # TABLE 1 — Fair comparison: different inputs, suited architectures
    # ----------------------------------------------------------------
    pixel_vs_rle = {
        "CNN (raw pixels)"     : cnn_result,
        "ResNet-18 (ImageNet)" : resnet_result,
        "RLE-Conv1D (ours)"    : rle_result,
    }

    print("\n" + "=" * 85)
    print("  TABLE 1 — INPUT REPRESENTATION COMPARISON")
    print("  (each model uses its natural input + suited architecture)")
    print("=" * 85)
    print(f"{'Model':<25}  {'Input':<12}  {'AUC':>6}  "
          f"{'RecN@0.50':>9}  {'F1@best':>7}  {'Thr':>5}  {'Time(s)':>8}")
    print("-" * 85)

    input_labels = {
        "CNN (raw pixels)"     : "pixels",
        "ResNet-18 (ImageNet)" : "pixels",
        "RLE-Conv1D (ours)"    : "RLE vec",
    }

    for name, r in pixel_vs_rle.items():
        marker = " ←" if name == "RLE-Conv1D (ours)" else ""
        print(f"{name:<25}  {input_labels[name]:<12}  {r['auc']:>6.4f}  "
              f"{r['rec_fix']:>9.4f}  {r['f1']:>7.4f}  "
              f"{r['thr']:>5.2f}  {r['time']:>8.2f}{marker}")

    print("=" * 85)
    best_auc  = max(pixel_vs_rle, key=lambda k: pixel_vs_rle[k]['auc'])
    best_recn = max(pixel_vs_rle, key=lambda k: pixel_vs_rle[k]['rec_fix'])
    print(f"  Best AUC       → {best_auc} ({pixel_vs_rle[best_auc]['auc']:.4f})")
    print(f"  Best Recall    → {best_recn} ({pixel_vs_rle[best_recn]['rec_fix']:.4f})")
    print(f"\n  Note: CNN and ResNet use raw 256×256 pixels as input.")
    print(f"        RLE-Conv1D uses 15,408-dim RLE feature vectors.")
    print(f"        This table compares the value of each representation.")

    # ----------------------------------------------------------------
    # TABLE 2 — RLE feature quality: same input, different classifiers
    # ----------------------------------------------------------------
    rle_classifiers = {
        **sk_results,
        "RLE-Conv1D (ours)" : rle_result,
    }

    print("\n" + "=" * 85)
    print("  TABLE 2 — CLASSIFIER COMPARISON ON RLE FEATURES")
    print("  (all models receive identical 15,408-dim RLE vectors)")
    print("=" * 85)
    print(f"{'Model':<25}  {'AUC':>6}  {'RecN@0.50':>9}  "
          f"{'F1@best':>7}  {'Thr':>5}  {'Time(s)':>8}  {'Note'}")
    print("-" * 85)

    notes = {
        "Random Forest"      : "ignores sequence order",
        "Gradient Boosting"  : "ignores sequence order",
        "SVM (RBF)"          : "PCA→150 dims first",
        "Logistic Regression": "linear boundary only",
        "KNN (k=7)"          : "PCA→150 dims first",
        "RLE-Conv1D (ours)"  : "respects run sequence ←",
    }

    for name, r in rle_classifiers.items():
        print(f"{name:<25}  {r['auc']:>6.4f}  {r['rec_fix']:>9.4f}  "
              f"{r['f1']:>7.4f}  {r['thr']:>5.2f}  "
              f"{r['time']:>8.2f}  {notes.get(name, '')}")

    print("=" * 85)
    best_auc2 = max(rle_classifiers, key=lambda k: rle_classifiers[k]['auc'])
    print(f"  Best AUC on RLE features → {best_auc2} "
          f"({rle_classifiers[best_auc2]['auc']:.4f})")
    print(f"\n  Interpretation: All models see identical RLE features.")
    print(f"  Conv1D wins because it understands that run positions")
    print(f"  are spatially ordered — adjacent runs are related.")
    print(f"  sklearn treats all 15,408 dimensions as independent.")
# =============================================================================
# CELL 20 — Entry point
# =============================================================================

if __name__ == "__main__":

    # First run:  FORCE_RECOMPUTE = True   (builds cache, slow)
    # After that: FORCE_RECOMPUTE = False  (loads cache, fast)
    FORCE_RECOMPUTE = True

    print("\n" + "="*70)
    print("CONFIGURATION:")
    print(f"  Device          : {DEVICE}")
    print(f"  Force recompute : {FORCE_RECOMPUTE}")
    print(f"  DataLoader workers: {NUM_DATALOADER_WORKERS}")
    print(f"  numba available : {HAS_NUMBA}")
    print(f"  Augmentations   : flip={AUG_PARAMS['flip_prob']}  rotate={AUG_PARAMS['rotate_prob']}")
    print("="*70)

    X_train, X_test, y_train, y_test = build_dataset(
        force_recompute = FORCE_RECOMPUTE,
        use_parallel    = False,
    )

    t0_rle                        = time.time()
    model, test_loader, criterion = train(X_train, X_test, y_train, y_test)
    rle_time                      = time.time() - t0_rle

    final_report(model, test_loader, criterion)

    _, _, rle_auc, y_true, y_prob = evaluate(model, test_loader, criterion, DEVICE)
    best_thr, best_f1 = best_threshold_sweep(y_true, y_prob)
    y_pred            = (y_prob >= best_thr).astype(int)
    tn, fp, fn, tp    = confusion_matrix(y_true, y_pred).ravel()
    y_pred_fixed      = (y_prob >= 0.50).astype(int)
    tn_f, fp_f, fn_f, tp_f = confusion_matrix(y_true, y_pred_fixed).ravel()

    rle_result = dict(
        acc      = ((y_pred == y_true).mean()),
        auc      = rle_auc,
        f1       = best_f1,
        recall_n = tp / (tp + fn),
        time     = rle_time,
        thr      = best_thr,
        acc_fix  = ((y_pred_fixed == y_true).mean()),
        f1_fix   = f1_score(y_true, y_pred_fixed),
        rec_fix  = tp_f / (tp_f + fn_f),
    )
    print(f"\nRLE → AUC: {rle_auc:.4f}  F1(best thr={best_thr:.2f}): {best_f1:.4f}  "
          f"F1(thr=0.50): {rle_result['f1_fix']:.4f}  "
          f"Recall_N: {rle_result['recall_n']:.4f}  "
          f"Train time: {rle_time:.2f}s")

    sk_results    = run_sklearn_benchmarks(X_train, X_test, y_train, y_test)
    cnn_result    = run_cnn_benchmark(BENIGN_DIR, MALIGNANT_DIR, NORMAL_DIR)
    resnet_result = run_resnet_benchmark(BENIGN_DIR, MALIGNANT_DIR, NORMAL_DIR)

    print_comparison(sk_results, cnn_result, rle_result, resnet_result)


CONFIGURATION:
  Device          : cuda
  Force recompute : True
  DataLoader workers: 0
  numba available : False
  Augmentations   : flip=0.0  rotate=0.0

EXTRACTING FEATURES FROM SCRATCH
Augmentation: flip=0.0  rotate=0.0

[1/4] Extracting from 623 training images...

  → Benign (cancer) train


  label=0 train: 100%|██████████| 349/349 [00:10<00:00, 32.73it/s]


  → Malignant (cancer) train


  label=0 train: 100%|██████████| 168/168 [00:05<00:00, 32.35it/s]


  → Normal train


...

  → Normal test


  label=1 test: 100%|██████████| 27/27 [00:00<00:00, 30.56it/s]



[2/4] Combining...

[3/4] Saving cache...

[4/4] Stats:
  Original train images : 623
  Actual train samples  : 623  (1.00x)
  Train shape: (623, 15408)  cancer=517  normal=106
  Test  shape: (157, 15408)
  Memory: 36.6 MB (train)  9.2 MB (test)

[Imbalance] strategy='smote'  loss='focal'
  Class weights — cancer(0): 0.603  normal(1): 2.939

  Class distribution before resampling: cancer(0)=517  normal(1)=106  (minority=class 1)
  [SMOTE disabled for RLE] using exact-fill repeat oversampling instead.
  Exact-fill oversample (+411 samples) → cancer(0)=517  normal(1)=517

Training on cuda  MAX_RUNS=18  SE_reduction=8  fused_dim=512  workers=0  AMP=on
Epoch   TrainLoss   TrainAcc   ValLoss   ValAcc      AUC
--------------------------------------------------------------


c:\Users\ARJUN\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\conv.py:306: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at ..\aten\src\ATen\native\Convolution.cpp:1032.)
  return F.conv1d(input, weight, bias, self.stride,


    1      0.2235     0.6625    0.1909   0.1783   0.5886 ✓ best
    2      0.1389     0.7021    0.1928   0.2930   0.6930 ✓ best
    3      0.1200     0.7437    0.1894   0.4713   0.7484 ✓ best
    4      0.0964     0.8008    0.3207   0.4013   0.6764
    5      0.0954     0.8143    0.1944   0.6306   0.6783
    6      0.0724     0.8723    0.3755   0.3057   0.7761 ✓ best
    7      0.0770     0.8607    0.4295   0.3376   0.7707
    8      0.0705     0.8704    0.5523   0.3057   0.8125 ✓ best
    9      0.0559     0.8849    0.3753   0.4204   0.7899
   10      0.0432     0.9120    0.5064   0.3694   0.7610
   11      0.0477     0.9284    0.3797   0.4331   0.7840
   12      0.0348     0.9381    0.2404   0.5732   0.8259 ✓ best
   13      0.0355     0.9497    0.2491   0.6051   0.7687
   14      0.0280     0.9536    0.2890   0.5860   0.8225
   15      0.0205     0.9662    0.2591   0.6497   0.7803
   16      0.0198     0.9652    0.2954   0.5350   0.8644 ✓ best
   17      0.0171     0.9710    0.2598 

In [5]:
# =============================================================================
# CELL 21 — Verification: Cross-validation, Confidence Intervals, Ablation
# =============================================================================

from sklearn.model_selection import StratifiedKFold
from scipy.stats import bootstrap as scipy_bootstrap
import warnings
warnings.filterwarnings('ignore')

# -----------------------------------------------------------------------------
# SECTION 1 — 5-Fold Stratified Cross-Validation
# -----------------------------------------------------------------------------

def run_cross_validation(X, y, n_splits=5, n_epochs=150, patience=40):
    """
    Proper stratified k-fold CV on the full dataset.
    Oversampling is done INSIDE each fold to prevent data leakage.
    """
    print("\n" + "="*65)
    print("SECTION 1 — 5-FOLD STRATIFIED CROSS-VALIDATION")
    print("="*65)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    fold_aucs    = []
    fold_recalls = []
    fold_f1s     = []
    fold_accs    = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n--- Fold {fold}/{n_splits} ---")

        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        # Oversample INSIDE fold — critical to avoid leakage
        X_bal, y_bal = apply_oversampling(X_tr, y_tr, IMBALANCE_STRATEGY)

        criterion = compute_loss_fn(y_tr)

        train_ds = RLEDataset(X_bal, y_bal)
        val_ds   = RLEDataset(X_val, y_val)

        sampler = make_weighted_sampler(y_bal)
        pin     = (DEVICE.type == "cuda")

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                                  sampler=sampler, pin_memory=pin)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                                  shuffle=False, pin_memory=pin)

        torch.manual_seed(SEED + fold)
        model     = RLEClassifier(num_classes=2).to(DEVICE)
        optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
        scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs)
        scaler    = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None

        best_auc, best_state  = 0.0, None
        epochs_no_improve     = 0

        for epoch in range(1, n_epochs + 1):
            train_epoch(model, train_loader, optimizer, criterion, DEVICE, scaler)
            _, _, va_auc, _, _ = evaluate(model, val_loader, criterion, DEVICE, scaler)
            scheduler.step()

            if va_auc > best_auc:
                best_auc          = va_auc
                epochs_no_improve = 0
                best_state        = {k: v.cpu().clone()
                                     for k, v in model.state_dict().items()}
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= patience:
                break

        model.load_state_dict(best_state)
        _, acc, auc, y_true_f, y_prob_f = evaluate(
            model, val_loader, criterion, DEVICE, scaler)

        best_thr, best_f1 = best_threshold_sweep(y_true_f, y_prob_f)
        y_pred_fixed      = (y_prob_f >= 0.50).astype(int)

        try:
            tn_, fp_, fn_, tp_ = confusion_matrix(y_true_f, y_pred_fixed).ravel()
            recall = tp_ / (tp_ + fn_) if (tp_ + fn_) > 0 else 0.0
        except ValueError:
            recall = 0.0

        fold_aucs.append(auc)
        fold_recalls.append(recall)
        fold_f1s.append(best_f1)
        fold_accs.append(acc)

        print(f"  Fold {fold} → AUC={auc:.4f}  Recall@0.50={recall:.4f}  "
              f"F1(best)={best_f1:.4f}  Acc={acc:.4f}  (stopped epoch {epoch})")

    print(f"\n{'='*65}")
    print(f"CROSS-VALIDATION SUMMARY (n={n_splits} folds)")
    print(f"{'='*65}")
    print(f"  AUC     : {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}  "
          f"[{np.min(fold_aucs):.4f} – {np.max(fold_aucs):.4f}]")
    print(f"  Recall  : {np.mean(fold_recalls):.4f} ± {np.std(fold_recalls):.4f}  "
          f"[{np.min(fold_recalls):.4f} – {np.max(fold_recalls):.4f}]")
    print(f"  F1      : {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}  "
          f"[{np.min(fold_f1s):.4f} – {np.max(fold_f1s):.4f}]")
    print(f"  Accuracy: {np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}")

    return {
        'auc_mean'   : np.mean(fold_aucs),
        'auc_std'    : np.std(fold_aucs),
        'recall_mean': np.mean(fold_recalls),
        'recall_std' : np.std(fold_recalls),
        'f1_mean'    : np.mean(fold_f1s),
        'f1_std'     : np.std(fold_f1s),
        'all_aucs'   : fold_aucs,
    }


# -----------------------------------------------------------------------------
# SECTION 2 — Bootstrap Confidence Intervals on held-out test set
# -----------------------------------------------------------------------------

def bootstrap_confidence_intervals(model, test_loader, criterion,
                                    n_bootstrap=1000, ci=0.95):
    """
    Bootstrap 95% CI for AUC, Recall, F1 on the fixed test set.
    Uses the already-trained best model from Cell 20.
    """
    print("\n" + "="*65)
    print("SECTION 2 — BOOTSTRAP CONFIDENCE INTERVALS (test set)")
    print(f"  n_bootstrap={n_bootstrap}  CI={int(ci*100)}%")
    print("="*65)

    _, _, _, y_true, y_prob = evaluate(model, test_loader, criterion, DEVICE)

    def stat_auc(yt, yp, axis):
        results = []
        for i in range(yt.shape[1] if yt.ndim > 1 else 1):
            col_yt = yt[:, i] if yt.ndim > 1 else yt
            col_yp = yp[:, i] if yp.ndim > 1 else yp
            try:
                results.append(roc_auc_score(col_yt, col_yp))
            except Exception:
                results.append(0.5)
        return np.array(results)

    # Manual bootstrap (more control than scipy for paired data)
    rng = np.random.default_rng(SEED)
    n   = len(y_true)

    boot_aucs    = []
    boot_recalls = []
    boot_f1s     = []

    for _ in range(n_bootstrap):
        idx = rng.choice(n, size=n, replace=True)
        yt  = y_true[idx]
        yp  = y_prob[idx]

        # Skip bootstrap samples with only one class
        if len(np.unique(yt)) < 2:
            continue

        boot_aucs.append(roc_auc_score(yt, yp))

        yp_fixed = (yp >= 0.50).astype(int)
        try:
            tn_, fp_, fn_, tp_ = confusion_matrix(yt, yp_fixed).ravel()
            boot_recalls.append(tp_ / (tp_ + fn_) if (tp_ + fn_) > 0 else 0.0)
        except ValueError:
            boot_recalls.append(0.0)

        boot_f1s.append(f1_score(yt, yp_fixed, zero_division=0))

    alpha = 1 - ci
    lo, hi = alpha / 2, 1 - alpha / 2

    auc_ci    = (np.quantile(boot_aucs,    lo), np.quantile(boot_aucs,    hi))
    recall_ci = (np.quantile(boot_recalls, lo), np.quantile(boot_recalls, hi))
    f1_ci     = (np.quantile(boot_f1s,     lo), np.quantile(boot_f1s,     hi))

    print(f"\n  Metric         Point Est.    {int(ci*100)}% CI")
    print(f"  {'─'*50}")
    print(f"  AUC            {roc_auc_score(y_true, y_prob):.4f}        "
          f"({auc_ci[0]:.4f},  {auc_ci[1]:.4f})")

    y_fixed = (y_prob >= 0.50).astype(int)
    try:
        tn_, fp_, fn_, tp_ = confusion_matrix(y_true, y_fixed).ravel()
        pt_recall = tp_ / (tp_ + fn_)
    except Exception:
        pt_recall = 0.0

    print(f"  Recall@0.50    {pt_recall:.4f}        "
          f"({recall_ci[0]:.4f},  {recall_ci[1]:.4f})")
    print(f"  F1@0.50        {f1_score(y_true, y_fixed):.4f}        "
          f"({f1_ci[0]:.4f},  {f1_ci[1]:.4f})")

    print(f"\n  Interpretation:")
    if auc_ci[0] > 0.85:
        print(f"  ✓ AUC CI lower bound {auc_ci[0]:.4f} > 0.85 — result is robust")
    elif auc_ci[0] > 0.75:
        print(f"  ~ AUC CI lower bound {auc_ci[0]:.4f} — moderate confidence")
    else:
        print(f"  ✗ AUC CI lower bound {auc_ci[0]:.4f} — high variance, need more data")

    return {'auc_ci': auc_ci, 'recall_ci': recall_ci, 'f1_ci': f1_ci}


# -----------------------------------------------------------------------------
# SECTION 3 — Ablation Study
# -----------------------------------------------------------------------------

def run_ablation(X_train, X_test, y_train, y_test,
                 n_epochs=100, patience=30):
    """
    Tests 5 variants to prove each component contributes.
    Runs fast (fewer epochs) — purpose is relative comparison, not best score.
    """
    print("\n" + "="*65)
    print("SECTION 3 — ABLATION STUDY")
    print("="*65)

    X_bal, y_bal = apply_oversampling(X_train, y_train, IMBALANCE_STRATEGY)
    criterion    = compute_loss_fn(y_train)
    sampler      = make_weighted_sampler(y_bal)
    pin          = (DEVICE.type == "cuda")

    train_ds = RLEDataset(X_bal,  y_bal)
    test_ds  = RLEDataset(X_test, y_test)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              sampler=sampler, pin_memory=pin)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                              shuffle=False, pin_memory=pin)

    def _train_and_eval(model_variant, label):
        optimizer = AdamW(model_variant.parameters(), lr=LR, weight_decay=1e-3)
        scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs)
        scaler    = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None

        best_auc, best_state, no_imp = 0.0, None, 0
        for epoch in range(1, n_epochs + 1):
            train_epoch(model_variant, train_loader, optimizer,
                        criterion, DEVICE, scaler)
            _, _, auc, _, _ = evaluate(model_variant, test_loader,
                                       criterion, DEVICE, scaler)
            scheduler.step()
            if auc > best_auc:
                best_auc  = auc
                no_imp    = 0
                best_state = {k: v.cpu().clone()
                              for k, v in model_variant.state_dict().items()}
            else:
                no_imp += 1
            if no_imp >= patience:
                break

        model_variant.load_state_dict(best_state)
        _, acc, auc, y_true_a, y_prob_a = evaluate(
            model_variant, test_loader, criterion, DEVICE, scaler)
        _, best_f1 = best_threshold_sweep(y_true_a, y_prob_a)

        y_fixed = (y_prob_a >= 0.50).astype(int)
        try:
            tn_, fp_, fn_, tp_ = confusion_matrix(y_true_a, y_fixed).ravel()
            recall = tp_ / (tp_ + fn_)
        except Exception:
            recall = 0.0

        print(f"  {label:<35}  AUC={auc:.4f}  F1={best_f1:.4f}  "
              f"Recall@0.50={recall:.4f}  Acc={acc:.4f}")
        return auc

    print(f"\n  {'Variant':<35}  {'AUC':>7}  {'F1':>7}  {'Recall':>12}  {'Acc':>7}")
    print(f"  {'─'*75}")

    # Variant 1 — rows only (zero out col features)
    class RLERowOnly(RLEClassifier):
        def forward(self, x):
            x_row   = x[:, :ROW_FEAT].view(x.size(0), CROP_ROWS, MAX_RUNS * 2)
            seq_row = x_row.permute(0, 2, 1)
            y_row   = self.conv_main(seq_row) + self.proj(seq_row)
            y_row   = self.gap_main(y_row).squeeze(-1)
            y_row   = self.se_a(y_row)
            a_r = self.gap_par(self.par_a(seq_row)).squeeze(-1)
            b_r = self.gap_par(self.par_b(seq_row)).squeeze(-1)
            c_r = self.gap_par(self.par_c(seq_row)).squeeze(-1)
            d_r = self.gap_par(self.par_d(seq_row)).squeeze(-1)
            p_row = self.se_b(torch.cat([a_r, b_r, c_r, d_r], dim=1))
            # pad zeros for missing col half so FC shape stays 512
            zeros = torch.zeros_like(torch.cat([y_row, p_row], dim=1))
            fused = torch.cat([y_row, zeros[:, :y_row.size(1)],
                               p_row, zeros[:, :p_row.size(1)]], dim=1)
            return self.fc(fused)

    # Variant 2 — cols only
    class RLEColOnly(RLEClassifier):
        def forward(self, x):
            x_col   = x[:, ROW_FEAT:].view(x.size(0), IMAGE_SIZE, MAX_RUNS * 2)
            seq_col = x_col.permute(0, 2, 1)
            y_col   = self.conv_main(seq_col) + self.proj(seq_col)
            y_col   = self.gap_main(y_col).squeeze(-1)
            y_col   = self.se_a(y_col)
            a_c = self.gap_par(self.par_a(seq_col)).squeeze(-1)
            b_c = self.gap_par(self.par_b(seq_col)).squeeze(-1)
            c_c = self.gap_par(self.par_c(seq_col)).squeeze(-1)
            d_c = self.gap_par(self.par_d(seq_col)).squeeze(-1)
            p_col = self.se_b(torch.cat([a_c, b_c, c_c, d_c], dim=1))
            zeros = torch.zeros_like(torch.cat([y_col, p_col], dim=1))
            fused = torch.cat([zeros[:, :y_col.size(1)], y_col,
                               zeros[:, :p_col.size(1)], p_col], dim=1)
            return self.fc(fused)

    # Variant 3 — no SE blocks
    class RLENoSE(RLEClassifier):
        def forward(self, x):
            x_row = x[:, :ROW_FEAT].view(x.size(0), CROP_ROWS,  MAX_RUNS * 2)
            x_col = x[:, ROW_FEAT:].view(x.size(0), IMAGE_SIZE, MAX_RUNS * 2)
            seq_row = x_row.permute(0, 2, 1)
            seq_col = x_col.permute(0, 2, 1)
            y_row = self.gap_main(
                self.conv_main(seq_row) + self.proj(seq_row)).squeeze(-1)
            y_col = self.gap_main(
                self.conv_main(seq_col) + self.proj(seq_col)).squeeze(-1)
            # skip se_a
            a_r = self.gap_par(self.par_a(seq_row)).squeeze(-1)
            b_r = self.gap_par(self.par_b(seq_row)).squeeze(-1)
            c_r = self.gap_par(self.par_c(seq_row)).squeeze(-1)
            d_r = self.gap_par(self.par_d(seq_row)).squeeze(-1)
            p_row = torch.cat([a_r, b_r, c_r, d_r], dim=1)  # skip se_b
            a_c = self.gap_par(self.par_a(seq_col)).squeeze(-1)
            b_c = self.gap_par(self.par_b(seq_col)).squeeze(-1)
            c_c = self.gap_par(self.par_c(seq_col)).squeeze(-1)
            d_c = self.gap_par(self.par_d(seq_col)).squeeze(-1)
            p_col = torch.cat([a_c, b_c, c_c, d_c], dim=1)  # skip se_b
            fused = torch.cat([y_row, y_col, p_row, p_col], dim=1)
            return self.fc(fused)

    # Variant 4 — no parallel branch (main branch only)
    class RLEMainOnly(RLEClassifier):
        def forward(self, x):
            x_row = x[:, :ROW_FEAT].view(x.size(0), CROP_ROWS,  MAX_RUNS * 2)
            x_col = x[:, ROW_FEAT:].view(x.size(0), IMAGE_SIZE, MAX_RUNS * 2)
            seq_row = x_row.permute(0, 2, 1)
            seq_col = x_col.permute(0, 2, 1)
            y_row = self.gap_main(
                self.conv_main(seq_row) + self.proj(seq_row)).squeeze(-1)
            y_row = self.se_a(y_row)
            y_col = self.gap_main(
                self.conv_main(seq_col) + self.proj(seq_col)).squeeze(-1)
            y_col = self.se_a(y_col)
            # zeros for parallel branch slots
            zeros = torch.zeros(x.size(0), PAR_FUSED_CHANNELS,
                                device=x.device)
            fused = torch.cat([y_row, y_col, zeros, zeros], dim=1)
            return self.fc(fused)

    # Variant 5 — full model (baseline for comparison)
    torch.manual_seed(SEED)
    auc_rows   = _train_and_eval(RLERowOnly(2).to(DEVICE),    "Rows only (no col RLE)")
    torch.manual_seed(SEED)
    auc_cols   = _train_and_eval(RLEColOnly(2).to(DEVICE),    "Cols only (no row RLE)")
    torch.manual_seed(SEED)
    auc_nose   = _train_and_eval(RLENoSE(2).to(DEVICE),       "Full model, no SE blocks")
    torch.manual_seed(SEED)
    auc_main   = _train_and_eval(RLEMainOnly(2).to(DEVICE),   "Main branch only (no parallel)")
    torch.manual_seed(SEED)
    auc_full   = _train_and_eval(RLEClassifier(2).to(DEVICE), "Full model (all components)")

    print(f"\n  Component contribution summary:")
    print(f"  Column RLE adds   : {auc_full - auc_rows:+.4f} AUC over rows-only")
    print(f"  Row RLE adds      : {auc_full - auc_cols:+.4f} AUC over cols-only")
    print(f"  SE blocks add     : {auc_full - auc_nose:+.4f} AUC over no-SE")
    print(f"  Parallel branch   : {auc_full - auc_main:+.4f} AUC over main-only")

    return {
        'rows_only'  : auc_rows,
        'cols_only'  : auc_cols,
        'no_se'      : auc_nose,
        'main_only'  : auc_main,
        'full_model' : auc_full,
    }


# -----------------------------------------------------------------------------
# SECTION 4 — Statistical significance vs CNN baseline
# -----------------------------------------------------------------------------

def significance_test(model, test_loader, criterion,
                       cnn_probs, n_bootstrap=2000):
    """
    Permutation bootstrap test: is RLE AUC significantly better than CNN?
    Requires cnn_probs saved from run_cnn_benchmark.
    """
    print("\n" + "="*65)
    print("SECTION 4 — STATISTICAL SIGNIFICANCE (RLE vs CNN)")
    print("="*65)

    _, _, _, y_true, rle_probs = evaluate(model, test_loader, criterion, DEVICE)

    rle_auc = roc_auc_score(y_true, rle_probs)
    cnn_auc = roc_auc_score(y_true, cnn_probs)
    observed_diff = rle_auc - cnn_auc

    rng   = np.random.default_rng(SEED)
    n     = len(y_true)
    count = 0

    for _ in range(n_bootstrap):
        idx      = rng.choice(n, size=n, replace=True)
        yt       = y_true[idx]
        if len(np.unique(yt)) < 2:
            continue
        diff = (roc_auc_score(yt, rle_probs[idx]) -
                roc_auc_score(yt, cnn_probs[idx]))
        if diff >= observed_diff:
            count += 1

    p_value = count / n_bootstrap

    print(f"\n  RLE AUC : {rle_auc:.4f}")
    print(f"  CNN AUC : {cnn_auc:.4f}")
    print(f"  Δ AUC   : {observed_diff:+.4f}")
    print(f"  p-value : {p_value:.4f}  (bootstrap, n={n_bootstrap})")

    if p_value < 0.05:
        print(f"  ✓ Significant (p<0.05) — RLE outperforms CNN baseline")
    elif p_value < 0.10:
        print(f"  ~ Marginal (p<0.10) — trend favors RLE, not conclusive")
    else:
        print(f"  ✗ Not significant (p={p_value:.4f}) — difference may be noise")

    return p_value


# -----------------------------------------------------------------------------
# ENTRY POINT — run all verification sections
# -----------------------------------------------------------------------------

if __name__ == "__main__":

    # Reload full dataset
    X_all = np.concatenate([X_train, X_test], axis=0)
    y_all = np.concatenate([y_train, y_test], axis=0)

    print("\n" + "="*65)
    print("VERIFICATION SUITE")
    print(f"  Total samples : {len(X_all)}")
    print(f"  Cancer(0)     : {int((y_all==0).sum())}")
    print(f"  Normal(1)     : {int((y_all==1).sum())}")
    print("="*65)

    # Section 1 — cross-validation (uses full dataset, ~5x training time)
    cv_results = run_cross_validation(X_all, y_all, n_splits=5)

    # Section 2 — bootstrap CI (uses existing trained model from Cell 20)
    ci_results = bootstrap_confidence_intervals(
        model, test_loader, criterion, n_bootstrap=1000)

    # Section 3 — ablation (uses train/test split, faster epochs)
    ablation_results = run_ablation(X_train, X_test, y_train, y_test)

    # Section 4 — significance vs CNN
    # requires cnn_result['probs'] — add this line to run_cnn_benchmark:
    # return dict(..., probs=y_prob)  and pass cnn_result['probs'] below
    # p_value = significance_test(model, test_loader, criterion, cnn_probs)

    # ---------------------------------------------------------------------
    # Final summary
    # ---------------------------------------------------------------------
    print("\n" + "="*65)
    print("VERIFICATION SUMMARY")
    print("="*65)
    print(f"  CV AUC          : {cv_results['auc_mean']:.4f} ± {cv_results['auc_std']:.4f}")
    print(f"  CV Recall       : {cv_results['recall_mean']:.4f} ± {cv_results['recall_std']:.4f}")
    print(f"  Bootstrap AUC CI: ({ci_results['auc_ci'][0]:.4f}, {ci_results['auc_ci'][1]:.4f})")
    print(f"  Best ablation Δ : full={ablation_results['full_model']:.4f}  "
          f"rows_only={ablation_results['rows_only']:.4f}  "
          f"no_se={ablation_results['no_se']:.4f}")

    


VERIFICATION SUITE
  Total samples : 780
  Cancer(0)     : 647
  Normal(1)     : 133

SECTION 1 — 5-FOLD STRATIFIED CROSS-VALIDATION

--- Fold 1/5 ---

  Class distribution before resampling: cancer(0)=517  normal(1)=107  (minority=class 1)
  [SMOTE disabled for RLE] using exact-fill repeat oversampling instead.
  Exact-fill oversample (+410 samples) → cancer(0)=517  normal(1)=517
  Class weights — cancer(0): 0.603  normal(1): 2.916
  Fold 1 → AUC=0.8272  Recall@0.50=0.8846  F1(best)=0.5185  Acc=0.6603  (stopped epoch 56)

--- Fold 2/5 ---

  Class distribution before resampling: cancer(0)=517  normal(1)=107  (minority=class 1)
  [SMOTE disabled for RLE] using exact-fill repeat oversampling instead.
  Exact-fill oversample (+410 samples) → cancer(0)=517  normal(1)=517
  Class weights — cancer(0): 0.603  normal(1): 2.916
  Fold 2 → AUC=0.7893  Recall@0.50=0.9231  F1(best)=0.5417  Acc=0.4615  (stopped epoch 50)

--- Fold 3/5 ---

  Class distribution before resampling: cancer(0)=518  no